## 加载图片

现在，让我们加载一张图片。

In [ ]:
from PIL import Image
import requests

url = "https://github.com/timojl/clipseg/blob/master/example_image.jpg?raw=true"
image = Image.open(requests.get(url, stream=True).raw)
image

## 加载模型

接下来，让我们从[hub](https://huggingface.co/CIDAS/clipseg-rd64-refined)加载模型和它的处理器

In [ ]:
from mindnlp.transformers import CLIPSegProcessor, CLIPSegForImageSegmentation

processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined")


## 为模型处理图片和文本

为了模型能够使用处理器，可以准备图片和一些提示词。

In [ ]:
prompts = ["a glass", "something to fill", "wood", "a jar"]

inputs = processor(text=prompts, images=[image] * len(prompts), padding="max_length", return_tensors="ms")

## 前向传播

接下来，让我们运行一轮前向传播并且可视化模型所做的预测

In [ ]:
import mindspore
import matplotlib.pyplot as plt
from mindnlp.core import ops
# predict

outputs = model(**inputs)

preds = outputs.logits.unsqueeze(1)


In [ ]:
# visualize prediction
_, ax = plt.subplots(1, 5, figsize=(15, 4))
[a.axis('off') for a in ax.flatten()]
ax[0].imshow(image)
[ax[i+1].imshow(ops.sigmoid(preds[i][0]).asnumpy()) for i in range(4)];
[ax[i+1].text(0, -15, prompts[i]) for i in range(4)];

可以看出，该模型能够以零样本的方式根据文本提示进行图像分割。很酷吧？

## 转换为二进制掩码

为了转换为二进制掩码，我从[此处](https://github.com/amrrs/stable-diffusion-prompt-inpainting)借用了一些逻辑。人们可以在预测掩码上应用 sigmoid 激活函数，并使用一些 OpenCV (cv2) 将其转换为二进制掩码。

In [ ]:
filename = f"mask.png"
# here we save the second mask
plt.imsave(filename,ops.sigmoid(preds[1][0]).asnumpy())

In [ ]:
import cv2

img2 = cv2.imread(filename)

In [ ]:
gray_image = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

(thresh, bw_image) = cv2.threshold(gray_image, 100, 255, cv2.THRESH_BINARY)

# fix color format
cv2.cvtColor(bw_image, cv2.COLOR_BGR2RGB)

Image.fromarray(bw_image)